# NB1: Data Collection

**Project 1**: Language as a Hidden Variable: Measuring Behavioral Divergence in Multilingual LLMs

## Purpose
Run all API calls. Collect all model responses. Save raw data.
This notebook does **NOTHING** except collect data. No metrics, no analysis.

## Prerequisites
- NB0 must be complete
- `prompts_master.csv` must be verified (30 rows, all translations filled)
- Groq API key set in Colab Secrets

## API Call Summary
| Phase | Calls |
|-------|-------|
| Main experiment: 30 prompts × 3 langs × 2 models | 180 |
| Variance baseline: 30 prompts × 3 runs × 2 models | 180 |
| Sanity check: 10 prompts × 2 pairs × 2 models | 40 |
| **Total** | **~400** |

## Outputs
- `raw_responses.csv` (**BACK THIS UP IMMEDIATELY**)
- `sanity_check_responses.csv`
- `collection_log.txt`

---
## [1.0] Setup

In [ ]:
!pip install -q groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 141.7/141.7 kB 3.5 MB/s eta 0:00:00


In [ ]:
import pandas as pd
import numpy as np
import time
import os
import json
from datetime import datetime
from groq import Groq
from google.colab import userdata, drive

# Mount Google Drive
drive.mount('/content/drive')

# Paths — UPDATE if different
DATA_DIR = '/content/drive/MyDrive/nlp_genai_cie3_2/data'
RESULTS_DIR = '/content/drive/MyDrive/nlp_genai_cie3_2/results'

# API setup
GROQ_API_KEY = userdata.get('GROQ_API_KEY')
client = Groq(api_key=GROQ_API_KEY)

print("Setup complete.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Setup complete.


---
## [1.1] Load Prompts and Define Constants

In [ ]:
# Load prompts
# df_prompts = pd.read_csv(os.path.join(DATA_DIR, 'prompts_master.csv'))
df_prompts = pd.read_csv(
    os.path.join(DATA_DIR, 'prompts_master_csv_utf.csv')
)
assert len(df_prompts) == 30, f"Expected 30 prompts, got {len(df_prompts)}"

# Verify no empty translations
for col in ['english', 'hindi', 'french']:
    nulls = df_prompts[col].isnull().sum()
    empties = (df_prompts[col].astype(str).str.strip() == '').sum()
    assert nulls + empties == 0, f"Column '{col}' has {nulls + empties} empty cells. Fill all translations before running!"

print(f"Loaded {len(df_prompts)} prompts. All translations present.")

# Constants — LOCKED, do not change
# MODELS = ["llama-3.1-70b-versatile", "mixtral-8x7b-32768"]
MODELS = [
    "llama-3.3-70b-versatile",   # updated Llama
    "openai/gpt-oss-20b",     # newer Mixtral variant
]
LANGUAGES = ["english", "hindi", "french"]
TEMPERATURE = 0.1
MAX_TOKENS = 512
TOP_P = 1.0

# Pre-registered sanity check prompts
SANITY_CHECK_PROMPTS = ["F02", "F07", "N01", "N05", "N09", "S01", "S04", "S06", "S08", "S10"]

# Checkpoint and log files
CHECKPOINT_FILE = os.path.join(DATA_DIR, 'checkpoint_responses.csv')
LOG_FILE = os.path.join(DATA_DIR, 'collection_log.txt')

print(f"Models: {MODELS}")
print(f"Languages: {LANGUAGES}")
print(f"Params: temp={TEMPERATURE}, max_tokens={MAX_TOKENS}, top_p={TOP_P}")

Loaded 30 prompts. All translations present.
Models: ['llama-3.3-70b-versatile', 'openai/gpt-oss-20b']
Languages: ['english', 'hindi', 'french']
Params: temp=0.1, max_tokens=512, top_p=1.0


---
## [1.2] API Call Helper Function

Includes retry logic and error logging.

In [ ]:
# Collection log
collection_log = []

def log_message(msg):
    """Log a message with timestamp."""
    timestamp = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
    entry = f"[{timestamp}] {msg}"
    collection_log.append(entry)
    print(entry)

def call_groq_api(model_name, prompt_text, max_retries=2, retry_delay=30):
    """
    Call Groq API with retry logic.
    Returns: (response_text, latency_ms) or (None, None) on failure.
    """
    for attempt in range(max_retries):
        try:
            start_time = time.time()
            response = client.chat.completions.create(
                model=model_name,
                messages=[{"role": "user", "content": prompt_text}],
                temperature=TEMPERATURE,
                max_tokens=MAX_TOKENS,
                top_p=TOP_P
            )
            latency = (time.time() - start_time) * 1000
            reply = response.choices[0].message.content
            return reply, round(latency, 2)
        except Exception as e:
            log_message(f"API ERROR (attempt {attempt+1}/{max_retries}): model={model_name}, error={str(e)}")
            if attempt < max_retries - 1:
                log_message(f"Retrying in {retry_delay}s...")
                time.sleep(retry_delay)

    return None, None

def save_checkpoint(results, filepath):
    """Save checkpoint to CSV."""
    pd.DataFrame(results).to_csv(filepath, index=False)
    print(f"Checkpoint saved: {len(results)} responses -> {filepath}")

print("Helper functions defined.")

Helper functions defined.


---
## [1.3] Main Data Collection Loop

30 prompts × 3 languages × 2 models = **180 API calls**

Estimated time: ~10-15 minutes (with 2s delay between calls).

In [ ]:
main_results = []
response_id_counter = 1
failed_calls = []

log_message("=== STARTING MAIN DATA COLLECTION ===")
total_calls = len(df_prompts) * len(LANGUAGES) * len(MODELS)
call_count = 0

for model_name in MODELS:
    log_message(f"--- Model: {model_name} ---")
    for _, row in df_prompts.iterrows():
        prompt_id = row['prompt_id']
        category = row['category']

        for language in LANGUAGES:
            prompt_text = str(row[language]).strip()
            call_count += 1

            print(f"[{call_count}/{total_calls}] {model_name} | {prompt_id} | {language}", end=" ")

            response_text, latency = call_groq_api(model_name, prompt_text)

            if response_text is not None:
                main_results.append({
                    'response_id': response_id_counter,
                    'prompt_id': prompt_id,
                    'category': category,
                    'model': model_name,
                    'language': language,
                    'response_text': response_text,
                    'response_length': len(response_text),
                    'api_latency_ms': latency,
                    'timestamp': datetime.now().isoformat(),
                    'run_id': 'main_1'
                })
                response_id_counter += 1
                print(f"OK ({latency:.0f}ms, {len(response_text)} chars)")
            else:
                failed_calls.append({
                    'prompt_id': prompt_id,
                    'model': model_name,
                    'language': language,
                    'phase': 'main'
                })
                log_message(f"FAILED: {prompt_id} | {model_name} | {language}")
                print("FAILED")

            # Rate limiting delay
            time.sleep(2)

            # Checkpoint every 30 calls
            if len(main_results) % 30 == 0 and len(main_results) > 0:
                save_checkpoint(main_results, CHECKPOINT_FILE)

log_message(f"Main collection complete. {len(main_results)} responses collected. {len(failed_calls)} failures.")
save_checkpoint(main_results, CHECKPOINT_FILE)

---
## [1.4] Variance Baseline Collection (English Only, 3 Runs)

30 prompts × 3 runs × 2 models = **180 API calls** (English only)

This establishes the intra-language noise floor.

In [ ]:
baseline_results = []

log_message("=== STARTING VARIANCE BASELINE COLLECTION ===")
total_baseline_calls = len(df_prompts) * 3 * len(MODELS)
baseline_call_count = 0

for model_name in MODELS:
    log_message(f"--- Baseline Model: {model_name} ---")
    for _, row in df_prompts.iterrows():
        prompt_id = row['prompt_id']
        category = row['category']
        prompt_text = str(row['english']).strip()

        for run_num in range(1, 4):  # 3 runs
            baseline_call_count += 1
            run_id = f"baseline_{run_num}"

            print(f"[{baseline_call_count}/{total_baseline_calls}] {model_name} | {prompt_id} | run {run_num}", end=" ")

            response_text, latency = call_groq_api(model_name, prompt_text)

            if response_text is not None:
                baseline_results.append({
                    'response_id': response_id_counter,
                    'prompt_id': prompt_id,
                    'category': category,
                    'model': model_name,
                    'language': 'english',
                    'response_text': response_text,
                    'response_length': len(response_text),
                    'api_latency_ms': latency,
                    'timestamp': datetime.now().isoformat(),
                    'run_id': run_id
                })
                response_id_counter += 1
                print(f"OK ({latency:.0f}ms)")
            else:
                failed_calls.append({
                    'prompt_id': prompt_id,
                    'model': model_name,
                    'language': 'english',
                    'phase': f'baseline_run{run_num}'
                })
                log_message(f"FAILED BASELINE: {prompt_id} | {model_name} | run {run_num}")
                print("FAILED")

            time.sleep(2)

            if len(baseline_results) % 30 == 0 and len(baseline_results) > 0:
                save_checkpoint(
                    main_results + baseline_results,
                    CHECKPOINT_FILE
                )

log_message(f"Baseline collection complete. {len(baseline_results)} responses collected.")

---
## [1.5] Sanity Check Collection

10 pre-registered prompts × 2 language pairs × 2 models = **40 API calls**

For each pair, sends both responses to the LLM and asks for semantic equivalence judgment.

In [14]:
sanity_results = []

# Build a lookup from main_results for easy access
main_df = pd.read_csv(CHECKPOINT_FILE)

SANITY_CHECK_PROMPT_TEMPLATE = """Below are two responses to the same question, written in different languages.
Translate them both to English if needed, then judge:
Are these two responses semantically equivalent in meaning and intent?
Answer with exactly one word: YES, PARTIAL, or NO.

Response 1 (English):
{english_response}

Response 2 ({target_language}):
{target_response}"""

# Use llama-3.1-70b-versatile as the sanity check judge (per guide Section 5)
# SANITY_JUDGE_MODEL = "llama-3.1-70b-versatile"
SANITY_JUDGE_MODEL = "llama-3.3-70b-versatile"

log_message("=== STARTING SANITY CHECK COLLECTION ===")
sanity_call_count = 0
total_sanity = len(SANITY_CHECK_PROMPTS) * 2 * len(MODELS)  # 2 language pairs

for prompt_id in SANITY_CHECK_PROMPTS:
    for model_name in MODELS:
        # Get English response from main collection
        en_mask = (
            (main_df['prompt_id'] == prompt_id) &
            (main_df['model'] == model_name) &
            (main_df['language'] == 'english') &
            (main_df['run_id'] == 'main_1')
        )

        en_rows = main_df[en_mask]
        if len(en_rows) == 0:
            log_message(f"SKIP SANITY: No English response for {prompt_id}/{model_name}")
            continue
        # Ensure response_text is a string, even if it's NaN
        english_response = str(en_rows.iloc[0]['response_text'])

        for target_lang, lang_pair in [('hindi', 'EN-HI'), ('french', 'EN-FR')]:
            sanity_call_count += 1

            # Get target language response
            tgt_mask = (
                (main_df['prompt_id'] == prompt_id) &
                (main_df['model'] == model_name) &
                (main_df['language'] == target_lang) &
                (main_df['run_id'] == 'main_1')
            )

            tgt_rows = main_df[tgt_mask]
            if len(tgt_rows) == 0:
                log_message(f"SKIP SANITY: No {target_lang} response for {prompt_id}/{model_name}")
                continue
            # Ensure response_text is a string, even if it's NaN
            target_response = str(tgt_rows.iloc[0]['response_text'])

            # Build sanity check prompt
            check_prompt = SANITY_CHECK_PROMPT_TEMPLATE.format(
                english_response=english_response[:1000],  # Truncate if very long
                target_language=target_lang.capitalize(),
                target_response=target_response[:1000]
            )

            print(f"[{sanity_call_count}/{total_sanity}] Sanity: {prompt_id} | {model_name} | {lang_pair}", end=" ")

            judgment_text, latency = call_groq_api(SANITY_JUDGE_MODEL, check_prompt)

            if judgment_text is not None:
                # Extract judgment (first word)
                judgment_clean = judgment_text.strip().split()[0].upper() if judgment_text.strip() else 'ERROR'
                # Normalize to YES/PARTIAL/NO
                if judgment_clean not in ['YES', 'PARTIAL', 'NO']:
                    log_message(f"Non-standard judgment: '{judgment_text.strip()}' for {prompt_id}/{lang_pair}")
                    judgment_clean = judgment_text.strip()[:20]

                sanity_results.append({
                    'check_id': sanity_call_count,
                    'prompt_id': prompt_id,
                    'language_pair': lang_pair,
                    'model': model_name,
                    'english_response': english_response,
                    'target_response': target_response,
                    'llm_judgment': judgment_clean,
                    'llm_raw_response': judgment_text.strip()
                })
                print(f"-> {judgment_clean}")
            else:
                log_message(f"FAILED SANITY: {prompt_id} | {model_name} | {lang_pair}")
                print("FAILED")

            time.sleep(2)

log_message(f"Sanity check complete. {len(sanity_results)} judgments collected.")

[2026-04-08 05:19:56] === STARTING SANITY CHECK COLLECTION ===
[1/40] Sanity: F02 | llama-3.3-70b-versatile | EN-HI -> PARTIAL
[2/40] Sanity: F02 | llama-3.3-70b-versatile | EN-FR -> PARTIAL
[3/40] Sanity: F02 | openai/gpt-oss-20b | EN-HI [2026-04-08 05:20:01] API ERROR (attempt 1/2): model=llama-3.3-70b-versatile, error=Error code: 429 - {'error': {'message': 'Rate limit reached for model `llama-3.3-70b-versatile` in organization `org_01knnm302xefgrss6641qhcs89` service tier `on_demand` on tokens per day (TPD): Limit 100000, Used 99539, Requested 1383. Please try again in 13m16.608s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}
[2026-04-08 05:20:01] Retrying in 30s...


KeyboardInterrupt: 

In [ ]:
pd.DataFrame(sanity_results).to_csv("sanity_results.csv", index=False)

---
## [1.6] Save All Data

In [15]:
# Combine main + baseline into one raw_responses file

# all_responses = main_results + baseline_results
# df_raw = pd.DataFrame(all_responses)
df_raw = pd.read_csv(CHECKPOINT_FILE)

raw_path = os.path.join(DATA_DIR, 'raw_responses.csv')
df_raw.to_csv(raw_path, index=False)
log_message(f"Saved raw_responses.csv: {len(df_raw)} total responses")

df_sanity = pd.read_csv('/content/drive/MyDrive/nlp_genai_cie3_2/sanity_results.csv')


# # Save sanity check responses
# df_sanity = pd.DataFrame(sanity_results)
# sanity_path = os.path.join(DATA_DIR, 'sanity_check_responses.csv')
# df_sanity.to_csv(sanity_path, index=False)
# log_message(f"Saved sanity_check_responses.csv: {len(df_sanity)} judgments")

# Save collection log
log_path = os.path.join(DATA_DIR, 'collection_log.txt')
with open(log_path, 'w', encoding='utf-8') as f:
    f.write('\n'.join(collection_log))
log_message(f"Saved collection_log.txt")

# Print summary
print("\n" + "=" * 60)
print("DATA COLLECTION SUMMARY")
print("=" * 60)
print(f"Main experiment responses: {len(main_results)}")
print(f"  Expected: {30 * 3 * 2} (30 prompts × 3 languages × 2 models)")
print(f"Baseline responses: {len(baseline_results)}")
print(f"  Expected: {30 * 3 * 2} (30 prompts × 3 runs × 2 models)")
print(f"Sanity check judgments: {len(sanity_results)}")
print(f"  Expected: {10 * 2 * 2} (10 prompts × 2 pairs × 2 models)")
print(f"Failed calls: {len(failed_calls)}")
if failed_calls:
    print("Failed call details:")
    for fc in failed_calls:
        print(f"  {fc}")
print("=" * 60)

[2026-04-08 05:22:55] Saved raw_responses.csv: 360 total responses
[2026-04-08 05:22:56] Saved collection_log.txt

DATA COLLECTION SUMMARY
Main experiment responses: 180
  Expected: 180 (30 prompts × 3 languages × 2 models)
Baseline responses: 180
  Expected: 180 (30 prompts × 3 runs × 2 models)
Sanity check judgments: 2
  Expected: 40 (10 prompts × 2 pairs × 2 models)
Failed calls: 0


In [17]:
import pandas as pd
import os

# Load your freshly collected data
raw_path = os.path.join(DATA_DIR, 'raw_responses.csv')
df_raw = pd.read_csv(raw_path)

# Filter for the main experiment (180 responses)
# We don't usually need to label the variance baseline runs
df_main = df_raw[df_raw['run_id'] == 'main_1'].copy()

# Select columns needed for the template
template = df_main[['response_id', 'prompt_id', 'model', 'language']].copy()

# Add empty columns for your team to fill
template['annotator1_label'] = ""
template['annotator2_label'] = ""
template['consensus_label'] = ""

# Save as the label file
template.to_csv(os.path.join(DATA_DIR, 'refusal_labels.csv'), index=False)
print(f"Template generated with {len(template)} rows.")


Template generated with 180 rows.


---
## [1.7] Verify Data Completeness

Check for any missing prompt-language-model combinations.

In [16]:
# Check main experiment completeness
print("=== MAIN EXPERIMENT COMPLETENESS CHECK ===")
main_only = df_raw[df_raw['run_id'] == 'main_1']
missing_main = []

for model_name in MODELS:
    for _, row in df_prompts.iterrows():
        for language in LANGUAGES:
            mask = (main_only['prompt_id'] == row['prompt_id']) & \
                   (main_only['model'] == model_name) & \
                   (main_only['language'] == language)
            if mask.sum() == 0:
                missing_main.append({
                    'prompt_id': row['prompt_id'],
                    'model': model_name,
                    'language': language
                })

if missing_main:
    print(f"MISSING {len(missing_main)} combinations:")
    for m in missing_main:
        print(f"  {m}")
    print("\nRe-run these before proceeding to NB2!")
else:
    print("All 180 main experiment combinations present. COMPLETE.")

# Check baseline completeness
print("\n=== BASELINE COMPLETENESS CHECK ===")
baseline_only = df_raw[df_raw['run_id'].str.startswith('baseline')]
baseline_counts = baseline_only.groupby(['prompt_id', 'model']).size()
incomplete_baselines = baseline_counts[baseline_counts < 3]

if len(incomplete_baselines) > 0:
    print(f"Incomplete baselines ({len(incomplete_baselines)}):")
    print(incomplete_baselines)
else:
    print("All baseline runs (3 per prompt-model) present. COMPLETE.")

print("\n>>> If everything is COMPLETE, proceed to Phase 4 (Manual Annotation) <<<")
print(">>> Then proceed to NB2 after refusal_labels.csv is filled <<<")

=== MAIN EXPERIMENT COMPLETENESS CHECK ===
All 180 main experiment combinations present. COMPLETE.

=== BASELINE COMPLETENESS CHECK ===
All baseline runs (3 per prompt-model) present. COMPLETE.

>>> If everything is COMPLETE, proceed to Phase 4 (Manual Annotation) <<<
>>> Then proceed to NB2 after refusal_labels.csv is filled <<<
